In [9]:
import pandas as pd

# Load cleaned datasets with engineered metrics
sold = pd.read_csv('../data/02_intermediate/CRMLSSold_Cleaned.csv', low_memory=False)
listings = pd.read_csv('../data/02_intermediate/CRMLSListing_Cleaned.csv', low_memory=False)

print(f"Sold: {sold.shape[0]:,} rows, {sold.shape[1]} columns")
print(f"Listings: {listings.shape[0]:,} rows, {listings.shape[1]} columns")

Sold: 467,728 rows, 77 columns
Listings: 467,728 rows, 76 columns


In [10]:

# Keep original record count
original_count = len(sold)

# -----------------------------
# Business Rule Flags
# -----------------------------
sold['Invalid_ClosePrice'] = sold['ClosePrice'] <= 0
sold['Invalid_LivingArea'] = sold['LivingArea'] <= 0
sold['Invalid_DaysOnMarket'] = sold['DaysOnMarket'] < 0

# -----------------------------
# Function to Create IQR Flags
# -----------------------------
def create_iqr_flag(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    flag_column = f"{column}_Outlier"

    dataframe[flag_column] = (
        (dataframe[column] < lower) |
        (dataframe[column] > upper)
    )

    return dataframe

# -----------------------------
# Create Outlier Flags
# -----------------------------
for col in ['ClosePrice', 'LivingArea', 'DaysOnMarket']:
    sold = create_iqr_flag(sold, col)

# -----------------------------
# Save Full Flagged Dataset; goes to 02_intermediate
# -----------------------------
# sold.to_csv(
#     "../data/02_intermediate/CRMLSSold_Flagged.csv",
#     index=False
# )
# -----------------------------
# Create Filtered Dataset; goes to 03_processed
# -----------------------------
filtered_sold = sold[
    (~sold['Invalid_ClosePrice']) &
    (~sold['Invalid_LivingArea']) &
    (~sold['Invalid_DaysOnMarket']) &
    (~sold['ClosePrice_Outlier']) &
    (~sold['LivingArea_Outlier']) &
    (~sold['DaysOnMarket_Outlier'])
]

# filtered_sold.to_csv(
#     "../data/03_processed/CRMLSSold_Filtered.csv",
#     index=False
# )

# ----------------------------- 
# Comparison Statistics
# -----------------------------
print("\nDATASET SIZE")
print(f"Original Records: {len(sold)}")
print(f"Filtered Records: {len(filtered_sold)}")
print(f"Records Removed: {len(sold) - len(filtered_sold)}")

print("\nMEDIAN COMPARISON")

for col in ['ClosePrice', 'LivingArea', 'DaysOnMarket']:
    original_median = sold[col].median()
    filtered_median = filtered_sold[col].median()

    print(f"\n{col}")
    print(f"Original Median: {original_median:,.2f}")
    print(f"Filtered Median: {filtered_median:,.2f}")


DATASET SIZE
Original Records: 467728
Filtered Records: 397342
Records Removed: 70386

MEDIAN COMPARISON

ClosePrice
Original Median: 820,000.00
Filtered Median: 785,000.00

LivingArea
Original Median: 1,650.00
Filtered Median: 1,578.00

DaysOnMarket
Original Median: 17.00
Filtered Median: 15.00


In [11]:

# Keep original record count
original_count = len(listings)

# -----------------------------
# Business Rule Flags
# -----------------------------
listings['Invalid_ClosePrice'] = listings['ClosePrice'] <= 0
listings['Invalid_LivingArea'] = listings['LivingArea'] <= 0
listings['Invalid_DaysOnMarket'] = listings['DaysOnMarket'] < 0

# -----------------------------
# Function to Create IQR Flags
# -----------------------------
def create_iqr_flag(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    flag_column = f"{column}_Outlier"

    dataframe[flag_column] = (
        (dataframe[column] < lower) |
        (dataframe[column] > upper)
    )

    return dataframe

# -----------------------------
# Create Outlier Flags
# -----------------------------
for col in ['ClosePrice', 'LivingArea', 'DaysOnMarket']:
    listings = create_iqr_flag(listings, col)

# -----------------------------
# Save Full Flagged Dataset
# -----------------------------
# listings.to_csv(
#     "../data/02_intermediate/CRMLSListing_Flagged.csv",
#     index=False
# )
# -----------------------------
# Create Filtered Dataset
# -----------------------------
filtered_listings = listings[
    (~listings['Invalid_ClosePrice']) &
    (~listings['Invalid_LivingArea']) &
    (~listings['Invalid_DaysOnMarket']) &
    (~listings['ClosePrice_Outlier']) &
    (~listings['LivingArea_Outlier']) &
    (~listings['DaysOnMarket_Outlier'])
]

# filtered_listings.to_csv(
#     "../data/03_processed/CRMLSListing_Filtered.csv",
#     index=False
# )

# -----------------------------
# Comparison Statistics
# -----------------------------
print("\nDATASET SIZE")
print(f"Original Records: {len(listings)}")
print(f"Filtered Records: {len(filtered_listings)}")
print(f"Records Removed: {len(listings) - len(filtered_listings)}")

print("\nMEDIAN COMPARISON")

for col in ['ClosePrice', 'LivingArea', 'DaysOnMarket']:
    original_median = listings[col].median()
    filtered_median = filtered_listings[col].median()

    print(f"\n{col}")
    print(f"Original Median: {original_median:,.2f}")
    print(f"Filtered Median: {filtered_median:,.2f}")


DATASET SIZE
Original Records: 467728
Filtered Records: 397342
Records Removed: 70386

MEDIAN COMPARISON

ClosePrice
Original Median: 820,000.00
Filtered Median: 785,000.00

LivingArea
Original Median: 1,650.00
Filtered Median: 1,578.00

DaysOnMarket
Original Median: 17.00
Filtered Median: 15.00
